In [21]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.memory import MemorySaver # Store memory in RAM



In [13]:
from langgraph.graph.message import add_messages # So any new chat will add not last one removed
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    


In [14]:
load_dotenv()

True

In [15]:


llm = ChatGroq(model="openai/gpt-oss-120b")

def chat_node(state: ChatState):

    # Take user query
    messages = state['messages']

    # send to llm
    response = llm.invoke(messages)

    # response store in state
    return {'messages': [response]}

In [22]:
checkpointer = MemorySaver()
graph = StateGraph(ChatState)

# Add nodes
graph.add_node('chat_node',chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node',END)

chatbot = graph.compile(checkpointer=checkpointer)


In [18]:
initial_state = {
    'messages': [HumanMessage(content='what is the capital of india')]
}

chatbot.invoke(initial_state)['messages'][-1].content

'The capital of India is **New\u202fDelhi**.'

In [23]:
thread_id = '1'

while True:

    user_message = input('Type here: ')

    print('user:', user_message)

    if user_message.strip().lower() in ['exit','quit','bye']:
        break

    config = {'configurable': {'thread_id': thread_id}}

    response = chatbot.invoke({'messages':[HumanMessage(content=user_message)]}, config=config)

    print('AI:', response['messages'][-1].content)

    # Here chatbot cannot have memory power, now we use percistance

user: hii
AI: Hello! How can I help you today?
user: my name is kuldip
AI: Nice to meet you, Kuldip! How can I assist you today?
user: could you tell me my name ?
AI: Your name is Kuldip.
user: nice bro
AI: Glad you’re happy with that, Kuldip! Anything else you’d like to chat about or need help with?
user: could you give me code for facorial number
AI: Sure! Here’s a simple implementation of a factorial function in Python, along with a few usage examples.

```python
def factorial(n: int) -> int:
    """
    Compute the factorial of a non‑negative integer n.

    Args:
        n (int): The number to compute the factorial of. Must be >= 0.

    Returns:
        int: n! (the factorial of n)

    Raises:
        ValueError: If n is negative.
    """
    if n < 0:
        raise ValueError("Factorial is not defined for negative numbers.")
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result


# ------------------------------
# Example usage
# -------------------